# Manual llama.cpp CUDA Setup (Windows)

Run this notebook one time to prepare `llama.cpp` + `llama-server` for GPU usage before running the app.

This notebook does not start the FastAPI app. It only prepares tools and optionally starts `llama-server` for testing.

In [ ]:
import os
import shutil
import subprocess
import time
from pathlib import Path

import requests

ROOT = Path.cwd().resolve()
LLAMA_CPP_DIR = (ROOT / "llama.cpp").resolve()
BUILD_DIR = LLAMA_CPP_DIR / "build"

print(f"ROOT={ROOT}")
print(f"LLAMA_CPP_DIR={LLAMA_CPP_DIR}")

In [ ]:
# Check required tools first
required_tools = ["git", "cmake"]
missing = [tool for tool in required_tools if shutil.which(tool) is None]
if missing:
    raise RuntimeError(f"Missing required tools on PATH: {missing}. Install them first.")
print("All required tools found on PATH.")

In [ ]:
# Clone llama.cpp if missing
if not LLAMA_CPP_DIR.exists():
    print("Cloning llama.cpp ...")
    subprocess.run(["git", "clone", "https://github.com/ggerganov/llama.cpp.git", str(LLAMA_CPP_DIR)], check=True)
else:
    print("llama.cpp already exists. Skipping clone.")

In [ ]:
# Configure and build with CUDA
BUILD_DIR.mkdir(parents=True, exist_ok=True)

print("Configuring CMake (CUDA ON) ...")
subprocess.run(["cmake", "..", "-DGGML_CUDA=ON", "-DCMAKE_BUILD_TYPE=Release"], cwd=str(BUILD_DIR), check=True)

print("Building llama.cpp ...")
subprocess.run(["cmake", "--build", \
, "--config", "Release", "-j"], cwd=str(BUILD_DIR), check=True)

print("Build completed.")

In [ ]:
# Locate llama-server executable
candidates = [
    BUILD_DIR / "bin" / "Release" / "llama-server.exe",
    BUILD_DIR / "bin" / "llama-server.exe",
    BUILD_DIR / "bin" / "llama-server",
]
llama_server_exe = next((p for p in candidates if p.exists()), None)

if llama_server_exe is None:
    hits = [f for f in BUILD_DIR.rglob("llama-server*") if f.is_file()]
    llama_server_exe = hits[0] if hits else None

if llama_server_exe is None:
    raise FileNotFoundError("llama-server executable not found after build.")

print(f"llama-server: {llama_server_exe}")

## Optional: Start llama-server and verify health

Set `GGUF_MODEL_PATH` below to your local model path before running the next cell.

In [ ]:
# Optional runtime check
GGUF_MODEL_PATH = os.getenv("LOCAL_GGUF_PATH", "").strip()
HOST = "127.0.0.1"
PORT = 8081

if not GGUF_MODEL_PATH:
    print("Set LOCAL_GGUF_PATH env var (or edit GGUF_MODEL_PATH) before running server test.")
else:
    cmd = [
        str(llama_server_exe),
        "--model",
        GGUF_MODEL_PATH,
        "--host",
        HOST,
        "--port",
        str(PORT),
        "--ctx-size",
        "8192",
        "--n-gpu-layers",
        "999",
        "--threads",
        str(max(1, (os.cpu_count() or 4) - 1)),
    ]
    print("Starting llama-server for health check ...")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")

    health_ok = False
    for _ in range(90):
        time.sleep(1)
        try:
            r = requests.get(f"http://{HOST}:{PORT}/health", timeout=2)
            if r.status_code == 200:
                health_ok = True
                break
        except Exception:
            pass

    print("Health check:", "OK" if health_ok else "FAILED")
    if health_ok:
        print(f"Use this in app env: LLAMA_SERVER_URL=http://{HOST}:{PORT}/v1")

    # Stop test process. Comment out if you want to keep it running.
    proc.terminate()
    try:
        proc.wait(timeout=5)
    except Exception:
        proc.kill()